# Loss-Grid Computer: Train Missing Checkpoints

Trains same-family checkpoint variants needed for the `torch.compile` amortization experiment.

**Disconnect-safe**: each checkpoint is written to Drive and verified immediately after
its training run. Re-running the training cell skips any checkpoint already present in Drive.

Expected wall time on T4:
- `california_mlp` — ~1 min/seed
- `mnist_mlp` — ~3 min/seed
- `cifar10_row_gru` — ~15 min/seed
- Total (9 checkpoints) — ~57 min

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Must match the layout used by functional_eval_colab.ipynb.
DRIVE_ROOT = '/content/drive/MyDrive/loss-grid-experiments'
DRIVE_ASSETS_ROOT = f'{DRIVE_ROOT}/assets'

## 2. Load repo and install dependencies

In [ ]:
import os
import subprocess
from pathlib import Path

if 'DRIVE_ROOT' not in globals():
    DRIVE_ROOT = '/content/drive/MyDrive/loss-grid-experiments'
    DRIVE_ASSETS_ROOT = f'{DRIVE_ROOT}/assets'

REPO_DIR = Path('/content/loss-grid-computer')
DRIVE_REPO_DIR = Path(DRIVE_ROOT) / 'loss-grid-computer'

if REPO_DIR.exists():
    print(f'Repo already present at {REPO_DIR}')
elif DRIVE_REPO_DIR.exists():
    subprocess.check_call(['cp', '-r', str(DRIVE_REPO_DIR), str(REPO_DIR)])
    print(f'Repo copied from Drive to {REPO_DIR}')
else:
    subprocess.check_call([
        'git', 'clone',
        'https://github.com/hotz99/loss-grid-computer.git',
        str(REPO_DIR),
    ])
    print(f'Repo cloned to {REPO_DIR}')

os.chdir(REPO_DIR)
print(f'Working directory: {Path.cwd()}')

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision',
    'scikit-learn', 'pandas', 'numpy',
])
print('Dependencies installed.')

## 3. Verify CUDA runtime

In [ ]:
import os
import torch

assert torch.cuda.is_available(), 'No CUDA GPU. Change Runtime > Change runtime type > GPU.'
gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
print(f'GPU  : {gpu_name}')
print(f'VRAM : {props.total_memory / 1e9:.1f} GB')
print(f'CPU  : {os.cpu_count()} cores')
print(f'Torch: {torch.__version__}')

## 4. Configuration

In [ ]:
from pathlib import Path

SEEDS = [42, 99, 1337]   # three seeds to complement existing seed-0 checkpoints
DEVICE = 'cuda'

# Workload label -> (training script, checkpoint filename template)
WORKLOAD_SPECS = [
    ('california_mlp',  'training/train_california_mlp.py', 'california-mlp-{seed}.pkl'),
    ('mnist_mlp',       'training/train_mnist_mlp.py',      'mnist-mlp-{seed}.pkl'),
    ('cifar10_row_gru', 'training/train_row_gru.py',        'cifar10-row-gru-{seed}.pkl'),
]

DRIVE_ASSETS = Path(DRIVE_ASSETS_ROOT)
DRIVE_ASSETS.mkdir(parents=True, exist_ok=True)

print(f'Seeds    : {SEEDS}')
print(f'Device   : {DEVICE}')
print(f'Drive    : {DRIVE_ASSETS}')

## 5. Status — what is already saved on Drive

In [ ]:
from pathlib import Path

DRIVE_ASSETS = Path(DRIVE_ASSETS_ROOT)
print(f'{'Checkpoint':<35}  {'Size (KB)':>10}  Status')
print('-' * 58)
todo = []
for label, script, pattern in WORKLOAD_SPECS:
    for seed in SEEDS:
        name = pattern.format(seed=seed)
        p = DRIVE_ASSETS / name
        if p.exists():
            print(f'{name:<35}  {p.stat().st_size / 1e3:>10.0f}  on Drive')
        else:
            print(f'{name:<35}  {"":>10}  MISSING')
            todo.append((label, script, pattern, seed))
print()
print(f'To train: {len(todo)}   Already on Drive: {len(SEEDS) * len(WORKLOAD_SPECS) - len(todo)}')

## 6. Train missing checkpoints

Iterates each missing `(workload, seed)` pair. After each training run:
1. The checkpoint is explicitly copied to `DRIVE_ASSETS_ROOT`.
2. The Drive copy is verified with `torch.load`.

Re-running this cell is safe — completed checkpoints on Drive are skipped.

In [ ]:
import shutil
import subprocess
import sys
import torch
from datetime import datetime, timezone
from pathlib import Path


def now() -> str:
    return datetime.now(timezone.utc).strftime('%H:%M:%S')


def stream_run(cmd: list[str]) -> int:
    """Run cmd, stream stdout+stderr line-by-line, return exit code."""
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in iter(proc.stdout.readline, ''):
        print(line, end='', flush=True)
    proc.stdout.close()
    return proc.wait()


DRIVE_ASSETS = Path(DRIVE_ASSETS_ROOT)
REPO_DIR = Path('/content/loss-grid-computer')
failures = []

plan = [
    (label, script, pattern, seed)
    for label, script, pattern in WORKLOAD_SPECS
    for seed in SEEDS
    if not (DRIVE_ASSETS / pattern.format(seed=seed)).exists()
]
total = len(plan)
print(f'[{now()}] {total} checkpoint(s) to train\n')

for index, (label, script, pattern, seed) in enumerate(plan, start=1):
    ckpt_name = pattern.format(seed=seed)
    local_out = REPO_DIR / 'assets' / ckpt_name
    drive_out = DRIVE_ASSETS / ckpt_name

    print(f'[{now()}] ({index}/{total}) START  {label}  seed={seed}')
    print(f'          output -> {local_out}')
    print(f'          drive  -> {drive_out}')
    print('-' * 72, flush=True)

    cmd = [
        sys.executable, str(REPO_DIR / script),
        '--seed', str(seed),
        '--output', str(local_out),
        '--device', DEVICE,
    ]
    exit_code = stream_run(cmd)

    print('-' * 72)
    if exit_code != 0:
        print(f'[{now()}] ({index}/{total}) FAILED  {label}  seed={seed}  exit={exit_code}')
        failures.append(ckpt_name)
        continue

    if not local_out.exists():
        print(f'[{now()}] ({index}/{total}) FAILED  {ckpt_name} not found after training')
        failures.append(ckpt_name)
        continue

    # Explicit Drive copy — belt and suspenders alongside any symlink.
    shutil.copy2(str(local_out), str(drive_out))

    # Verify the Drive copy is loadable.
    try:
        state = torch.load(drive_out, map_location='cpu', weights_only=True)
        size_kb = drive_out.stat().st_size / 1e3
        print(
            f'[{now()}] ({index}/{total}) SAVED   {ckpt_name}'
            f'  {size_kb:.0f} KB  {len(state)} tensors  -> Drive OK'
        )
    except Exception as exc:
        print(f'[{now()}] ({index}/{total}) ERROR   Drive copy unloadable: {exc}')
        failures.append(ckpt_name)

    print(flush=True)

print('=' * 72)
if failures:
    print(f'[{now()}] DONE  {total - len(failures)}/{total} succeeded')
    for name in failures:
        print(f'  FAILED: {name}')
    raise RuntimeError(f'{len(failures)} checkpoint(s) failed — see above.')
else:
    print(f'[{now()}] DONE  all {total} checkpoints saved to Drive')

## 7. Final verification

In [ ]:
import torch
from pathlib import Path

DRIVE_ASSETS = Path(DRIVE_ASSETS_ROOT)
print(f'{'Checkpoint':<35}  {'Size (KB)':>10}  Tensors  Status')
print('-' * 66)
missing = []
for label, script, pattern in WORKLOAD_SPECS:
    for seed in SEEDS:
        name = pattern.format(seed=seed)
        p = DRIVE_ASSETS / name
        if not p.exists():
            print(f'{name:<35}  {"":>10}  {"":>7}  MISSING')
            missing.append(name)
            continue
        try:
            state = torch.load(p, map_location='cpu', weights_only=True)
            print(f'{name:<35}  {p.stat().st_size / 1e3:>10.0f}  {len(state):>7}  ok')
        except Exception as exc:
            print(f'{name:<35}  {"":>10}  {"":>7}  ERROR: {exc}')
            missing.append(name)

print()
total = len(SEEDS) * len(WORKLOAD_SPECS)
if missing:
    print(f'WARNING: {len(missing)}/{total} checkpoint(s) missing or unloadable.')
else:
    print(f'All {total} checkpoints verified on Drive.')